In [1]:
from pydantic import BaseModel, EmailStr, ValidationError
import json
from typing import Optional


In [2]:
from typing import List


class ResumeParser(BaseModel):
    name: str
    email: EmailStr
    phone: str
    skills: List[str]
    experience: List[str]
    education:List[str]

In [3]:
from email.mime import text
import re
def parse_resume(text: str):
    errors = []
    # lines = text.splitlines()

    #name
    name =text.strip().splitlines()[0].strip() 

    #Email
    email_match = re.search(r'[\w\.-]+@[\w-]+\.[a-zA-Z]{2,}', text)
    if not email_match:
        errors.append("Email not found")

    #Phone
    phone_match= re.search(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', text)
    if not phone_match:
        errors.append("Phone number not found")
    
    # Skills
    skills=[]
    in_skills_section = False

    for line in text.splitlines():
        if re.match(r'Skills', line.strip(), re.IGNORECASE):
            in_skills_section = True
            continue
        if in_skills_section:
            if line.strip() == "" :
                break
            skills.append(line.strip())
    
    #Experience
    experience = []
    in_experience_section = False
    for line in text.splitlines():
        if re.match(r'Experience', line.strip(), re.IGNORECASE):
            in_experience_section = True
            continue
        if in_experience_section:
            if line.strip() == "":
                break
            experience.append(line.strip())
    
    #Education
    #Education
    education = []
    in_education_section = False

    for line in text.splitlines():
        if re.match(r'Education', line.strip(), re.IGNORECASE):
            in_education_section = True
            continue

        if in_education_section:
            if line.strip() == "":
                break
            education.append(line.strip())
            

    #Validate with Pydantic
    try:
        resume = ResumeParser(
            name=name,
            email=email_match.group(0) if email_match else None,
            phone=phone_match.group(0) if phone_match else None,
            skills=skills if skills else None,
            experience=experience if experience else None,
            education=education if education else None
        )
        return {"success": True, "data": resume.model_dump(),"errors":[]}
    except ValidationError as e:
        for error in e.errors():
            errors.append(f"field '{error['loc'][0]}', message: {error['msg']}")
        return {"success": False, "data": None, "errors": errors}

In [4]:
example_resume_text = """John Doe
john@dec.in
123-456-7890
Skills
"Python" "Data Analysis" "Machine Learning"
Experience
"company": "ABC Corp", "role": "Data Scientist", "duration": "2 years"
"company": "XYZ Inc", "role": "Data Analyst", "duration": "3 years"
Education
"degree": "B.Sc. in Computer Science", "institution": "University A", "year": "2015"
"""

In [5]:
res=parse_resume(example_resume_text)
print(json.dumps(res, indent=4))

{
    "success": true,
    "data": {
        "name": "John Doe",
        "email": "john@dec.in",
        "phone": "123-456-7890",
        "skills": [
            "\"Python\" \"Data Analysis\" \"Machine Learning\"",
            "Experience",
            "\"company\": \"ABC Corp\", \"role\": \"Data Scientist\", \"duration\": \"2 years\"",
            "\"company\": \"XYZ Inc\", \"role\": \"Data Analyst\", \"duration\": \"3 years\"",
            "Education",
            "\"degree\": \"B.Sc. in Computer Science\", \"institution\": \"University A\", \"year\": \"2015\""
        ],
        "experience": [
            "\"company\": \"ABC Corp\", \"role\": \"Data Scientist\", \"duration\": \"2 years\"",
            "\"company\": \"XYZ Inc\", \"role\": \"Data Analyst\", \"duration\": \"3 years\"",
            "Education",
            "\"degree\": \"B.Sc. in Computer Science\", \"institution\": \"University A\", \"year\": \"2015\""
        ],
        "education": [
            "\"degree\": \"B.Sc.

In [6]:
bad_resume_text = """John Doe
john-dec.in
123-456-7890
Skills
Python
Education
B.Tech
"""

In [7]:
res1=parse_resume(bad_resume_text)
print(json.dumps(res1, indent=4))

{
    "success": false,
    "data": null,
    "errors": [
        "Email not found",
        "field 'email', message: Input should be a valid string",
        "field 'experience', message: Input should be a valid list"
    ]
}


In [8]:
bad_resume_text = """John Doe
john-dec.in
123-456-7890
Education
B.Tech
"""

In [9]:
res1=parse_resume(bad_resume_text)
print(json.dumps(res1, indent=4))

{
    "success": false,
    "data": null,
    "errors": [
        "Email not found",
        "field 'email', message: Input should be a valid string",
        "field 'skills', message: Input should be a valid list",
        "field 'experience', message: Input should be a valid list"
    ]
}
